# Harmonize scored-event labels of an .edf EEG/PSG database

This notebook lets you **visualize** the scored-event configurations present in a heterogeneous
.edf database (events annotated during sleep scoring and exported by Profusion/Compumedics:
apnea, hypopnea, arousals, limb movements, PLM, SpO2 desaturation…) and **harmonize** their raw
labels to a single canonical vocabulary.

It returns a JSON file `config_param/event_remap.json` (a flat python dict
`{raw_label: canonical_label}`, with `null` for labels you choose to ignore) that downstream tools
(epoch rejection, event-epoch viewer) can use.

---
**To use this notebook, interact with the widgets and read the output below. You first have to
select a database to make the widgets appear.**

Sections:
1. Select your study folder and scan the events
1bis. *(optional)* Check that the `*_event_xml.csv` and the `*.edf.XML` agree
2. Event configurations found
3. Harmonize the labels
4. Preview & save the JSON
5. Verify

The events are read from `*_event_xml.csv` first and, when absent, from the `<ScoredEvents>` of the
`*.edf.XML` (Profusion `CMPStudyConfig`).

---
last update 19/06/2026, YN


## Definitions (imports, helpers, event loaders, scan/save/verify logic)

In [ ]:
# Imports
try:
    import os
    import re
    import json
    import traceback
    import xml.etree.ElementTree as ET
    from pathlib import Path
    from collections import Counter, OrderedDict
    import pandas as pd
    import ipywidgets as widgets
    from ipyfilechooser import FileChooser
    from IPython.display import display, HTML, clear_output
except ImportError as e:
    print("⚠️ Error: ", e)
else:
    print("✅ Packages and functions successfully imported!")


# ---- generic helpers (shared idioms with select&remap_channels_edf) ----
def _load_json_lenient(path):
    """Load a JSON file, tolerating a single trailing comma before a closing } or ]
    (a common hand-edit mistake): strict parse first, repair only on failure."""
    with open(path, encoding="utf-8") as f:
        text = f.read()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return json.loads(re.sub(r",(\s*[}\]])", r"\1", text))


def print_in_scrollable_box(text, height=300, font_size="12px"):
    display(HTML(f'<pre style="overflow-y:scroll; height:{height}px; border:1px solid black; '
                 f'padding:10px; font-size:{font_size};">{text}</pre>'))


# ---- canonical event vocabulary (editable suggestions only) ----
# Compumedics/Profusion raw labels -> harmonized snake_case canonical labels.
# Arousal subtypes are kept (clinically meaningful); laterality is dropped for limbs.
DEFAULT_EVENT_MAPPING = {
    "obstructive apnea":   "apnea_obstructive",
    "central apnea":       "apnea_central",
    "mixed apnea":         "apnea_mixed",
    "hypopnea":            "hypopnea",
    "spo2 desaturation":   "spo2_desaturation",
    "arousal (aro res)":   "arousal_respiratory",
    "arousal (aro spont)": "arousal_spontaneous",
    "arousal (aro plm)":   "arousal_limb",
    "arousal (aro limb)":  "arousal_limb",
    "arousal ()":          "arousal",
    "arousal":             "arousal",
    "limb movement (left)":  "limb_movement",
    "limb movement (right)": "limb_movement",
    "limb movement":         "limb_movement",
    "plm (left)":  "plm",
    "plm (right)": "plm",
    "plm":         "plm",
}
CANONICAL_VOCAB = sorted(set(DEFAULT_EVENT_MAPPING.values()))


def suggest_canonical(raw):
    """Suggest a canonical label for a raw event name (empty string if unknown)."""
    key = raw.strip().lower()
    if key in DEFAULT_EVENT_MAPPING:
        return DEFAULT_EVENT_MAPPING[key]
    # tolerate a trailing laterality marker, e.g. "Limb Movement (Left)"
    stripped = re.sub(r"\s*\((left|right)\)\s*$", "", key).strip()
    return DEFAULT_EVENT_MAPPING.get(stripped, "")


# ---- event companion loading: CSV first, XML (<ScoredEvents>) fallback ----
def event_companion_paths(edf_path):
    """Return (csv_path_or_None, xml_path_or_None) for an EDF stem."""
    edf_path = Path(edf_path)
    csv = edf_path.with_name(f"{edf_path.stem}_event_xml.csv")
    csv = csv if csv.exists() else None
    xml = None
    for cand in (f"{edf_path.name}.XML", f"{edf_path.name}.xml"):
        p = edf_path.with_name(cand)
        if p.exists():
            xml = p
            break
    return csv, xml


def load_events_from_csv(csv_path):
    """Parse a Compumedics *_event_xml.csv -> list of (name, start, duration)."""
    df = pd.read_csv(csv_path)
    for col in ("Name", "Start", "Duration"):
        if col not in df.columns:
            raise ValueError(f"missing column '{col}' in {Path(csv_path).name}")
    events = []
    for _, row in df.iterrows():
        events.append((str(row["Name"]).strip(), float(row["Start"]), float(row["Duration"])))
    return events


def load_events_from_xml(xml_path):
    """Parse the <ScoredEvents> of a Profusion CMPStudyConfig .edf.XML
    -> list of (name, start, duration). <Input> is ignored (absent from the CSV)."""
    root = ET.parse(xml_path).getroot()
    events = []
    for se in root.iter("ScoredEvent"):
        name_el = se.find("Name")
        if name_el is None or name_el.text is None:
            continue
        start_el = se.find("Start")
        dur_el = se.find("Duration")
        start = float(start_el.text) if (start_el is not None and start_el.text) else float("nan")
        dur = float(dur_el.text) if (dur_el is not None and dur_el.text) else float("nan")
        events.append((name_el.text.strip(), start, dur))
    return events


def load_events(edf_path):
    """CSV-first, XML-fallback event loader.
    Returns (events_list, source) with source in {'csv', 'xml'} or (None, None) when
    neither companion is usable. events_list = list of (name, start, duration)."""
    csv, xml = event_companion_paths(edf_path)
    if csv is not None:
        try:
            return load_events_from_csv(csv), "csv"
        except Exception:
            pass  # fall through to the XML
    if xml is not None:
        try:
            return load_events_from_xml(xml), "xml"
        except Exception:
            pass
    return None, None


def load_existing_mapping(folder):
    """Read the existing config_param/event_remap.json (lenient), {} if absent/unreadable."""
    p = Path(folder) / "config_param" / "event_remap.json"
    if p.exists():
        try:
            return _load_json_lenient(p)
        except Exception:
            return {}
    return {}


def _events_multiset(events):
    """Multiset of (name_lower, round(start,3), round(dur,3)) for CSV/XML comparison."""
    return Counter((n.strip().lower(), round(s, 3), round(d, 3)) for (n, s, d) in events)


# ---- shared state filled by the scan ----
STATE = {}


## Build and display the interactive interface

In [ ]:
# ========================= Section banners =========================
section1 = widgets.HTML("""
<hr style="height:4px; background-color:black; border:none;">
<h2>1. Select your study folder and scan the events</h2>
<p>Pick the folder of your .edf database. Each .edf is expected to have a Compumedics/Profusion
event companion next to it: a <code>*_event_xml.csv</code> (read first) or, as a fallback,
the <code>&lt;ScoredEvents&gt;</code> of the <code>*.edf.XML</code>.
<br>&#x2022; Selecting the folder only refreshes the info line below.
<br>&#x2022; Click <b>Run scan</b> to read the events and list the configurations.
<br>&#x2022; "Skip labels already mapped" hides labels already present in an existing
<code>event_remap.json</code> (incremental harmonization when you add a new cohort).</p>
""")

section1bis = widgets.HTML("""
<hr style="height:4px; background-color:black; border:none;">
<h2>1bis. (Optional) Check CSV vs XML consistency</h2>
<p>Verifies that, for every file having <b>both</b> companions, the <code>*_event_xml.csv</code>
and the <code>&lt;ScoredEvents&gt;</code> of the <code>*.edf.XML</code> describe the same events
(name + start + duration, tolerance 1e-3 s). Writes <code>config_param/event_source_mismatch.tsv</code>.
This is opt-in because it forces reading both files for every EDF.</p>
""")

section2 = widgets.HTML("""
<hr style="height:4px; background-color:black; border:none;">
<h2>2. Event configurations found</h2>
<p>Files are grouped by their set of <b>unique</b> event labels. Two files with the same unique
labels share one configuration even if their event counts differ.</p>
""")

section3 = widgets.HTML("""
<hr style="height:4px; background-color:black; border:none;">
<h2>3. Harmonize the labels</h2>
<p>For each unique raw label, choose a harmonized (canonical) label. Defaults are pre-filled
when recognized; you can edit them freely.
<br>&#x2022; Tick <b>ignore</b> to drop a label (stored as <code>null</code>, excluded downstream).
<br>&#x2022; Then go to section 4 to preview &amp; save.</p>
""")

section4 = widgets.HTML("""
<hr style="height:4px; background-color:black; border:none;">
<h2>4. Preview &amp; save the JSON</h2>
<p>Builds a flat <code>{raw_label: canonical_label}</code> mapping and <b>merges</b> it into any
existing <code>config_param/event_remap.json</code> (labels mapped this session replace their old
value, all others are kept).</p>
""")

section5 = widgets.HTML("""
<hr style="height:4px; background-color:black; border:none;">
<h2>5. Verify</h2>
<p>Applies the saved mapping to every configuration and reports the resulting harmonized labels.
The verdict passes when no raw label is left unmapped (ignored labels count as handled).</p>
""")

# ========================= Section 1 widgets =========================
chooser = FileChooser(os.getcwd())
chooser.title = "<b>Choose your study folder</b>"
chooser.show_only_dirs = True

skip_existing = widgets.Checkbox(value=True, description="Skip labels already mapped",
                                 style={"description_width": "initial"})
existing_info = widgets.HTML(value="")
run_scan_button = widgets.Button(description="Run scan", button_style="success", icon="play")
out_scan = widgets.Output()

# Section 1bis
run_check_button = widgets.Button(description="Run CSV vs XML check", button_style="info", icon="check")
out_check = widgets.Output()

# Section 2 / 3 / 4 / 5 output zones
out_configs = widgets.Output()
out_harmonize = widgets.Output()
fname_text = widgets.Text(value="event_remap.json", description="File name:",
                          style={"description_width": "initial"}, layout=widgets.Layout(width="320px"))
preview_save_button = widgets.Button(description="Preview & Save", button_style="success", icon="save")
out_save = widgets.Output()
verify_scope = widgets.Dropdown(options=["All configurations", "Only configs with unmapped labels"],
                                value="All configurations", description="Show:",
                                style={"description_width": "initial"}, layout=widgets.Layout(width="380px"))
verify_button = widgets.Button(description="Run verification", button_style="success", icon="play")
out_verify = widgets.Output()


# ========================= Callbacks =========================
def _update_info(*_):
    """Refresh the info line when the folder changes (no scan)."""
    if not getattr(chooser, "selected_path", None):
        existing_info.value = ""
        return
    try:
        folder = Path(chooser.selected_path)
        edfs = [f for f in folder.rglob("*") if f.suffix.lower() == ".edf" and not f.name.startswith("._")]
        if not edfs:
            existing_info.value = '<small style="color:#888;">No EDF files found in selected folder.</small>'
            return
        existing = load_existing_mapping(folder)
        msg = f"<small>{len(edfs)} EDF file(s) found. "
        msg += (f"{len(existing)} label(s) already mapped in event_remap.json."
                if existing else "No existing event_remap.json yet.")
        existing_info.value = msg + "</small>"
    except Exception as e:
        existing_info.value = f'<small style="color:#c33;">{type(e).__name__}: {e}</small>'


def run_scan(_=None):
    for z in (out_scan, out_configs, out_harmonize, out_save, out_verify):
        z.clear_output()
    with out_scan:
        if not getattr(chooser, "selected_path", None):
            print("⚠️ Please select your study folder first.")
            return
        folder = Path(chooser.selected_path)
        edfs = sorted(f for f in folder.rglob("*")
                      if f.suffix.lower() == ".edf" and not f.name.startswith("._"))
        if not edfs:
            print("⚠️ No EDF files found in the selected folder.")
            return
        configs = {}          # frozenset(labels) -> [file_id]
        label_files = {}      # raw label -> set(file_id)
        label_counts = {}     # raw label -> total occurrences
        source_by_file = {}   # file_id -> 'csv'/'xml'
        failed = []           # (file_id, reason)
        n_with = 0
        for edf in edfs:
            fid = edf.stem
            try:
                events, source = load_events(edf)
            except Exception as e:
                failed.append((fid, f"{type(e).__name__}: {e}"))
                continue
            if events is None:
                failed.append((fid, "no readable event companion (.csv or .edf.XML)"))
                continue
            n_with += 1
            source_by_file[fid] = source
            names = [n for (n, _, _) in events]
            configs.setdefault(frozenset(names), []).append(fid)
            for nm in names:
                label_counts[nm] = label_counts.get(nm, 0) + 1
            for nm in set(names):
                label_files.setdefault(nm, set()).add(fid)
        STATE.update(folder=folder, configs=configs, label_files=label_files,
                     label_counts=label_counts, source_by_file=source_by_file,
                     failed=failed, existing=load_existing_mapping(folder))
        if failed:
            cfgdir = folder / "config_param"
            cfgdir.mkdir(exist_ok=True)
            pd.DataFrame(failed, columns=["file_id", "reason"]).to_csv(
                cfgdir / "failed_event_read.tsv", sep="\t", index=False)
        print(f"✅ Scanned {len(edfs)} EDF file(s): {n_with} with events, {len(failed)} failed.")
        print(f"   {len(label_files)} unique raw label(s), {len(configs)} distinct event configuration(s).")
        n_csv = sum(1 for s in source_by_file.values() if s == "csv")
        n_xml = sum(1 for s in source_by_file.values() if s == "xml")
        print(f"   event source: {n_csv} from CSV, {n_xml} from XML fallback.")
        if STATE["existing"]:
            print(f"   {len(STATE['existing'])} label(s) already in event_remap.json.")
        if failed:
            print(f"   ⚠ {len(failed)} file(s) without readable events "
                  f"— see config_param/failed_event_read.tsv")
    render_configs()
    render_harmonize()


def render_configs():
    with out_configs:
        clear_output()
        configs = STATE.get("configs", {})
        if not configs:
            return
        items = sorted(configs.items(), key=lambda kv: (-len(kv[1]), sorted(kv[0])))
        html = []
        for i, (labels, fids) in enumerate(items, 1):
            labs = sorted(labels)
            html.append(f"<h4>Configuration {i} &mdash; {len(fids)} file(s), "
                        f"{len(labs)} unique label(s)</h4>")
            html.append("<b>Labels:</b><br>" + "<br>".join(f"&#x2022; {l}" for l in labs))
            html.append(f"<details><summary>file ids ({len(fids)})</summary>"
                        + ", ".join(sorted(fids)) + "</details><hr>")
        display(HTML("".join(html)))
        lf, lc = STATE["label_files"], STATE["label_counts"]
        rows = sorted(lf.keys(), key=lambda l: (-len(lf[l]), l.lower()))
        df = pd.DataFrame([{"raw_label": l, "n_files": len(lf[l]), "n_occurrences": lc[l],
                            "suggested_canonical": suggest_canonical(l) or "(none)"} for l in rows])
        display(HTML("<h4>All unique raw labels</h4>"))
        display(HTML(df.to_html(index=False)))


def render_harmonize(*_):
    with out_harmonize:
        clear_output()
        lf = STATE.get("label_files", {})
        if not lf:
            return
        existing = STATE.get("existing", {})
        labels = sorted(lf.keys(), key=lambda l: (-len(lf[l]), l.lower()))
        if skip_existing.value:
            labels = [l for l in labels if l not in existing]
        STATE["rows"] = {}
        row_widgets = []
        if not labels:
            display(HTML("<i>All raw labels are already mapped (uncheck “Skip labels "
                         "already mapped” to edit them).</i>"))
            return
        header = widgets.HTML("<b>raw label</b> (files) &nbsp;&rarr;&nbsp; <b>canonical label</b>"
                              " &nbsp; <b>ignore</b>")
        for raw in labels:
            if raw in existing:
                val = existing[raw]
                combo_val, ignore_val = ("", True) if val is None else (val, False)
            else:
                combo_val, ignore_val = suggest_canonical(raw), False
            lab = widgets.HTML(f"<code>{raw}</code> <small style='color:#888'>({len(lf[raw])})</small>",
                               layout=widgets.Layout(width="320px"))
            combo = widgets.Combobox(value=combo_val, options=CANONICAL_VOCAB, ensure_option=False,
                                     placeholder="canonical label", disabled=ignore_val,
                                     layout=widgets.Layout(width="240px"))
            ign = widgets.Checkbox(value=ignore_val, description="ignore", indent=False,
                                   layout=widgets.Layout(width="90px"))

            def _toggle(change, c=combo):
                c.disabled = change["new"]
            ign.observe(_toggle, names="value")
            STATE["rows"][raw] = (combo, ign)
            row_widgets.append(widgets.HBox([lab, combo, ign]))
        box = widgets.VBox(row_widgets, layout=widgets.Layout(
            max_height="420px", overflow="auto", border="1px solid #ddd", padding="6px"))
        display(widgets.VBox([header, box]))


def on_preview_save(_=None):
    with out_save:
        clear_output()
        rows = STATE.get("rows", {})
        if not rows:
            print("⚠️ Run the scan (section 1) first.")
            return
        try:
            session, empties = {}, []
            for raw, (combo, ign) in rows.items():
                if ign.value:
                    session[raw] = None
                else:
                    v = combo.value.strip()
                    if v:
                        session[raw] = v
                    else:
                        empties.append(raw)
            folder = STATE["folder"]
            existing = load_existing_mapping(folder)   # reload fresh to merge
            merged = dict(existing)
            merged.update(session)
            merged = OrderedDict(sorted(merged.items(), key=lambda kv: kv[0].lower()))
            cfgdir = folder / "config_param"
            cfgdir.mkdir(exist_ok=True)
            out_json = cfgdir / (fname_text.value.strip() or "event_remap.json")
            with open(out_json, "w", encoding="utf-8") as f:
                json.dump(merged, f, indent=2, ensure_ascii=False)
            n_new = len(session)
            display(HTML(f"<p><b>✅ JSON saved here:</b> <code>{out_json}</code></p>"))
            display(HTML(f"<p style='color:#555'>{len(merged)} label(s) total "
                         f"(<b>{n_new}</b> added/updated this session); previous entries preserved.</p>"))
            if empties:
                display(HTML("<p style='color:#c33'>⚠ left unmapped (not saved): "
                             + ", ".join(f"<code>{e}</code>" for e in empties)
                             + " — set a canonical label or tick “ignore”.</p>"))
            preview = json.dumps(merged, indent=2, ensure_ascii=False)
            print_in_scrollable_box(preview.replace("<", "&lt;").replace(">", "&gt;"), height=260)
            display(HTML("<br><b>To load this file later:</b>"))
            display(HTML("<p><code>with open(path, encoding='utf-8') as f: event_map = json.load(f)</code></p>"))
            display(HTML("<p><code>canonical = event_map.get(raw_label)  # None = ignore</code></p>"))
        except Exception as e:
            display(HTML(f"<pre style='color:#c33'>{type(e).__name__}: {e}\n"
                         f"{traceback.format_exc()}</pre>"))


def run_verify(_=None):
    with out_verify:
        clear_output()
        configs = STATE.get("configs", {})
        if not configs:
            print("⚠️ Run the scan (section 1) first.")
            return
        try:
            folder = STATE["folder"]
            mapping = load_existing_mapping(folder)
            if not mapping:
                print("⚠️ No event_remap.json found yet — save it in section 4 first.")
                return
            items = sorted(configs.items(), key=lambda kv: (-len(kv[1]), sorted(kv[0])))
            any_unmapped = False
            html = []
            for i, (labels, fids) in enumerate(items, 1):
                labs = sorted(labels)
                unmapped = [l for l in labs if l not in mapping]
                canon = sorted({mapping[l] for l in labs if mapping.get(l) is not None})
                if unmapped:
                    any_unmapped = True
                if verify_scope.value.startswith("Only") and not unmapped:
                    continue
                html.append(f"<h4>Configuration {i} &mdash; {len(fids)} file(s)</h4>")
                html.append("<b>Harmonized labels:</b> "
                            + (", ".join(f"<code>{c}</code>" for c in canon) or "<i>none</i>"))
                if unmapped:
                    html.append("<br><span style='color:#c33'><b>Unmapped:</b> "
                                + ", ".join(f"<code>{u}</code>" for u in unmapped) + "</span>")
                html.append("<hr>")
            if not html:
                html.append("<i>Nothing to show for this scope.</i>")
            display(HTML("".join(html)))
            if any_unmapped:
                display(HTML("<p style='color:#c33'><b>❌ Some raw labels are still unmapped.</b> "
                             "Map them in section 3 and save again.</p>"))
            else:
                display(HTML("<p style='color:#178a17'><b>✅ All raw labels are mapped "
                             "(or explicitly ignored).</b></p>"))
        except Exception as e:
            display(HTML(f"<pre style='color:#c33'>{type(e).__name__}: {e}</pre>"))


def run_csvxml_check(_=None):
    with out_check:
        clear_output()
        if not getattr(chooser, "selected_path", None):
            print("⚠️ Please select your study folder first.")
            return
        folder = Path(chooser.selected_path)
        edfs = sorted(f for f in folder.rglob("*")
                      if f.suffix.lower() == ".edf" and not f.name.startswith("._"))
        if not edfs:
            print("⚠️ No EDF files found.")
            return
        rows = []
        for edf in edfs:
            fid = edf.stem
            csv, xml = event_companion_paths(edf)
            if csv is None or xml is None:
                rows.append(dict(file_id=fid, n_csv="", n_xml="", n_only_in_csv="", n_only_in_xml="",
                                 labels_only_in_csv="", labels_only_in_xml="",
                                 status="missing CSV" if csv is None else "missing XML"))
                continue
            try:
                ce = _events_multiset(load_events_from_csv(csv))
                xe = _events_multiset(load_events_from_xml(xml))
            except Exception as e:
                rows.append(dict(file_id=fid, n_csv="", n_xml="", n_only_in_csv="", n_only_in_xml="",
                                 labels_only_in_csv="", labels_only_in_xml="",
                                 status=f"error: {type(e).__name__}"))
                continue
            only_csv = ce - xe
            only_xml = xe - ce
            rows.append(dict(
                file_id=fid, n_csv=sum(ce.values()), n_xml=sum(xe.values()),
                n_only_in_csv=sum(only_csv.values()), n_only_in_xml=sum(only_xml.values()),
                labels_only_in_csv="; ".join(sorted({k[0] for k in only_csv})),
                labels_only_in_xml="; ".join(sorted({k[0] for k in only_xml})),
                status="match" if not only_csv and not only_xml else "MISMATCH"))
        df = pd.DataFrame(rows)
        cfgdir = folder / "config_param"
        cfgdir.mkdir(exist_ok=True)
        df.to_csv(cfgdir / "event_source_mismatch.tsv", sep="\t", index=False)
        n_match = int((df["status"] == "match").sum())
        n_mis = int((df["status"] == "MISMATCH").sum())
        n_other = len(df) - n_match - n_mis
        color = "#178a17" if (n_mis == 0 and n_other == 0) else "#c98a00"
        display(HTML(f"<p style='color:{color}'><b>{n_match} match</b>, {n_mis} mismatch, "
                     f"{n_other} missing-companion/error — of {len(df)} file(s).</p>"))
        display(HTML(f"<p style='color:#555'>Saved <code>{cfgdir / 'event_source_mismatch.tsv'}</code></p>"))
        show = df[df["status"] != "match"]
        if len(show):
            display(HTML("<b>Files needing attention:</b>"))
            display(HTML(show.to_html(index=False)))


# ========================= Wiring & layout =========================
chooser.register_callback(_update_info)
skip_existing.observe(lambda ch: render_harmonize(), names="value")
run_scan_button.on_click(run_scan)
run_check_button.on_click(run_csvxml_check)
preview_save_button.on_click(on_preview_save)
verify_button.on_click(run_verify)

ui_layout = widgets.VBox([
    section1, chooser, existing_info, skip_existing, run_scan_button, out_scan,
    section1bis, run_check_button, out_check,
    section2, out_configs,
    section3, out_harmonize,
    section4, widgets.HBox([fname_text, preview_save_button]), out_save,
    section5, widgets.HBox([verify_scope, verify_button]), out_verify,
])
display(ui_layout)
